# LAB 11 - Random Forest for Regression

In this lab we will be extending the previous lab about Decision trees and build a Regression model using Random Forest.

For simplicity, we will be using the same dataset as the previous lab (you can find it in ECLASS).

**IMPORTANT:** For this lab, if you haven't finished your code from last week's lab on Decision trees, you will have the option to use the sklearn implementation for a regression tree. However, this doesn't mean that you should skip the previous lab. This is just so that you don't get behind with the content and you don't spend all your time today working on the previous lab. 

In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

As mentioned before, use the Boston Housing data and prepare your train/val/test split as usual.

In [2]:
housing = pd.read_csv("../data/housing.csv")
display(housing.head())

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.000000,0.165514,0.067815,0.0,0.273696,0.455845,0.495675,0.238389,0.000000,0.188979,0.252496,0.693147,0.085884,3.218876
1,0.000236,0.000000,0.242302,0.0,0.159428,0.436962,0.578128,0.299335,0.042560,0.099811,0.440312,0.693147,0.186040,3.117950
2,0.000236,0.000000,0.242302,0.0,0.159428,0.527320,0.469617,0.299335,0.042560,0.099811,0.440312,0.688003,0.061533,3.575151
3,0.000293,0.000000,0.063050,0.0,0.139941,0.505947,0.365901,0.370559,0.083382,0.064658,0.500130,0.690281,0.032843,3.538057
4,0.000705,0.000000,0.063050,0.0,0.139941,0.523014,0.424170,0.370559,0.083382,0.064658,0.500130,0.693147,0.094708,3.616309


In [3]:
X = housing.iloc[:, :-1].values
y = housing["MEDV"].values

X_train, X_aux, y_train, y_aux = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_aux, y_aux, test_size=0.5)

## Exercise 1 -- Bootstrap

Also known as [bagging](https://en.wikipedia.org/wiki/Bootstrap_aggregating), this technique consists of making several samples with replacement of the original data, using each of the samples to train an estimator, and then aggregating the predictions using the average (this is also a type of model ensemble).

In [4]:
def bootstrap(X, num_bags=10):
    """
    Given a dataset and a number of bags,
    sample the dataset with replacement.
    
    This function does not return a copy
    of the datapoints, but a list of indices
    with compatible dimensionality
    
    Parameters
    ----------
    X : ndarray
        A dataset
    num_bags : int, default 10
        The number of bags to create
    
    Returns
    -------
    list of ndarray
        The list contains `num_bags` integer one-dimensional ndarrays.
        Each of these contains the indices corresponding to the 
        sampled datapoints in `X`
    
    Notes
    -----
    * The number of datapoints in each bach will
      match the number of datapoints in the given
      dataset.
    * The
    """
    rng = np.random.default_rng(0) # you can change the seed, or use 0 to replicate my results
    # Your code here

    tam = X.shape[0]
    indices = []
    for _ in range(num_bags):
        indices.append(rng.choice(tam, size=tam, replace=True))

    return indices
    

In [5]:
rng = np.random.default_rng(0)
X_small = rng.random(size=(100,2))
bags = bootstrap(X_small)
bags[0]

array([85, 63, 51, 26, 30,  4,  7,  1, 17, 81, 64, 91, 50, 60, 97, 72, 63,
       54, 55, 93, 27, 81, 67,  0, 39, 85, 55,  3, 76, 72, 84, 17,  8, 86,
        2, 54,  8, 29, 48, 42, 40,  2,  0, 12,  0, 67, 52, 64, 25, 61, 76,
       38, 46, 99, 80, 98, 37, 68, 95, 65, 84, 68, 70, 38, 87, 13, 57, 72,
       84, 52, 37, 31, 42, 48, 71, 88,  7, 93, 53, 35, 67, 57, 25, 32, 71,
       59, 50, 33, 76, 39, 32, 89, 26, 22, 71, 62,  4,  8, 37, 83])

## Exercise 2 -- Aggregation

The second part of bagging.

In [6]:
def aggregate_regression(preds):
    """
    Aggregate predictions by several estimators
    
    Parameters
    ----------
    preds : list of ndarray
        Predictions from multiple estimators.
        All ndarrays in this list should have the same
        dimensionality.
        
    Return
    ------
    ndarray
        The mean of the predictions
    """
    # Your code here

    np_preds = np.array(preds).T
    return np.mean(np_preds, axis=1)

## Exercise 3 -- Random Forest for regression

Using the functions you implemented above, it is now time to put all of them together to train several decision trees and then ensemble them to output a single prediction. For the random forest, however, we need to select a subset of features at each split on the decision tree. 

For this part, you can use the sklearn implementation of Decision trees for regression as your estimator for each set of features and bags. See below an example of how to do this, and always remember to check the necessary documentation when using an external function.

Some parameters you will have to set are: 
* num_features: number of features per estimator
* min_samples: min number of samples per leaf node
* max_depth: maximum depth of the decision tree (each estimator)
* num_estimators: number of decision trees you will create using each bag and random set of features

In [7]:
max_depth = 5
min_samples = 1
num_estimators = 10

In [8]:
# example of sklearn Decision tree
estimator = DecisionTreeRegressor(max_depth=max_depth)
estimator.fit(X, y)
estimator.predict(X)

array([3.34057739, 3.1382379 , 3.52327442, 3.52327442, 3.52327442,
       3.21380844, 3.04244879, 2.95714391, 2.95714391, 2.95714391,
       2.95714391, 3.04244879, 3.01757488, 3.04244879, 3.1382379 ,
       3.04244879, 3.21380844, 3.01757488, 3.04244879, 3.04244879,
       2.76937391, 3.04244879, 2.76937391, 2.76937391, 2.76937391,
       2.76937391, 2.95714391, 2.76937391, 3.1382379 , 3.1382379 ,
       2.76937391, 3.04244879, 2.65324196, 2.76937391, 2.76937391,
       3.04244879, 3.04244879, 3.04244879, 3.04244879, 3.34057739,
       3.52327442, 3.52327442, 3.21380844, 3.21380844, 3.04244879,
       3.04244879, 3.04244879, 2.95714391, 2.95714391, 3.15336475,
       3.04244879, 3.1382379 , 3.21380844, 3.04244879, 3.01757488,
       3.52327442, 3.21380844, 3.52327442, 3.21380844, 3.04244879,
       3.04244879, 2.95714391, 3.21380844, 3.38770766, 3.38770766,
       3.21380844, 3.04244879, 3.04244879, 3.04244879, 3.04244879,
       3.21380844, 3.04244879, 3.21380844, 3.21380844, 3.21380

In [9]:
## your code goes here:
class Forest:
    def __init__(self, max_depth=5, min_samples=1):
        self.max_depth = max_depth
        self.min_samples = min_samples

    def fit(self, X, y, num_estimators = 10):
        baggin_idx = bootstrap(X, num_estimators)
        self.estimators = []
        for bag in baggin_idx:
            est = DecisionTreeRegressor(max_depth=self.max_depth)
            est.fit(X[bag], y[bag])
            self.estimators.append(est)

    def predict(self, X):
        preds = []
        for est in self.estimators:
            preds.append(est.predict(X))

        return aggregate_regression(preds)

In [10]:
f = Forest(max_depth, min_samples)
f.fit(X_train, y_train, num_estimators)
pred_val = f.predict(X_val)
print('RMSE on valitation: ', np.mean((y_val - pred_val) ** 2) ** (1/2))

RMSE on valitation:  0.11045733982536857


In [11]:
X_train_val = np.vstack((X_train, X_val))
y_train_val = np.concat((y_train, y_val))

In [12]:
f = Forest(max_depth, min_samples)
f.fit(X_train_val, y_train_val, num_estimators)
pred_test = f.predict(X_test)
print('RMSE on testing', np.mean((y_test - pred_test) ** 2) ** (1/2))

RMSE on testing 0.20692154543043548
